# 02 — Unstructured → structured data

Turn messy files into queryable Spark DataFrames.

**Before running**, generate the practice files from the project root:

```powershell
python scripts/generate_unstructured_data.py
```

That writes three files into `data/generated/`:

| File | Messiness |
|------|-----------|
| `events.jsonl` | Nested JSON lines; some `props` are strings instead of objects; some users missing |
| `app.log` | Free-text logs in **two** slightly different formats |
| `support_tickets.csv` | CSV with pipe-delimited tags + free-text bodies that hide IDs |

Work top to bottom. Each exercise builds on the previous. Suggested approach for every task:

1. Load the raw data and look at a few rows.
2. Parse / extract fields into typed columns.
3. Answer the questions with normal DataFrame ops (`filter`, `groupBy`, etc.).

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("unstructured-to-structured")
    .master("local[*]")
    .getOrCreate()
)

DATA_DIR = "../data/generated"
print(f"Spark version: {spark.version}")

Spark version: 4.1.2


## Exercise A — Nested JSON event logs (`events.jsonl`)

Each line is one JSON event. Nesting means fields like `user.id` and `device.type` aren't flat columns yet.

### Goals
1. Read the file as JSON (hint: `spark.read.json(...)` works on `.jsonl`).
2. Flatten into columns: `event_id`, `ts`, `user_id`, `session`, `page`, `device_type`, `browser`, `referrer`, `cart_value`.
3. Parse `ts` into a real timestamp.
4. ~5% of rows store `props` as a **string** (JSON-inside-a-string). Detect those and parse them so `referrer` / `cart_value` aren't null just because of that mess.
5. Answer:
   - How many events have a missing `user_id`?
   - Top 5 pages by event count
   - Average `cart_value` by `device_type` (ignore null cart values)

In [3]:
# A0 — inspect the raw JSON (schema will show nested structs / odd types)
raw_events = spark.read.json(f"{DATA_DIR}/events.jsonl")
raw_events.printSchema()
raw_events.show(3, truncate=False)

root
 |-- device: struct (nullable = true)
 |    |-- browser: string (nullable = true)
 |    |-- type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- props: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- session: string (nullable = true)

+-----------------+---------+---------+------------------------------------------+--------------------+--------------+
|device           |event_id |page     |props                                     |ts                  |user          |
+-----------------+---------+---------+------------------------------------------+--------------------+--------------+
|{chrome, tablet} |evt-00001|/        |{"referrer": "email", "cart_value": null} |2024-06-17T02:17:33Z|{203, s-1791} |
|{safari, desktop}|evt-00002|/checkout|{"referrer": "google", "cart_value": null}|2024-06-11T19:08:29Z|{NULL, s-8104}|
|{edge, tabl

In [3]:
# A1 — YOUR TURN: flatten + clean into a queryable `events` DataFrame
# Hints:
#   F.col("user.id"), F.from_json(...), F.to_timestamp(...)
#   For stringified props, check the schema of `props` — it may be a mix.
#   One approach: if props is a string column, from_json it; if it's a struct, use it directly.
#   With this generator, Spark often infers `props` as STRING because of the dirty rows —
#   so from_json on the whole column is a good first try.

props_schema = T.StructType([
    T.StructField("referrer", T.StringType()),
    T.StructField("cart_value", T.DoubleType()),
])

# raw_events.select(F.from_json(F.col("props"), props_schema).referrer).show(5)

events = (raw_events.withColumn("props", F.from_json(F.col("props"), props_schema))
                    .select("device.*", "event_id", "page", "props.*",
                    F.to_timestamp(F.col("ts")), "user.*")
                    .withColumnRenamed("browser", "device_browser")
                    .withColumnRenamed("type", "device_type")
                    .withColumnRenamed("id", "user_id")
                    .withColumnRenamed("session", "user_session")
         )

    
events.printSchema()
events.show(5, truncate=False)

root
 |-- device_browser: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- cart_value: double (nullable = true)
 |-- to_timestamp(ts): timestamp (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)

+--------------+-----------+---------+---------+--------+----------+-------------------+-------+------------+
|device_browser|device_type|event_id |page     |referrer|cart_value|to_timestamp(ts)   |user_id|user_session|
+--------------+-----------+---------+---------+--------+----------+-------------------+-------+------------+
|chrome        |tablet     |evt-00001|/        |email   |NULL      |2024-06-16 22:17:33|203    |s-1791      |
|safari        |desktop    |evt-00002|/checkout|google  |NULL      |2024-06-11 15:08:29|NULL   |s-8104      |
|edge          |tablet     |evt-00003|/help    |google  |NULL    

In [56]:
# A2 — YOUR TURN: answer the three questions

missing_users = events.filter(F.col("user_id").isNull())
print("missing users: ", missing_users.count())

# Top 5 pages by event count
top_pages = events.groupBy(F.col("page")).count().orderBy(F.desc("count"))
top_pages.show(5)

# Average Cart Value by event type
avg_cart = (events.groupBy("device_type")
                .agg(F.avg("cart_value"))
                .alias("avg_cart")
           )
avg_cart.show()

missing users:  159
+---------+-----+
|     page|count|
+---------+-----+
|    /help|  355|
| /account|  342|
|    /cart|  336|
|/checkout|  332|
|/products|  329|
+---------+-----+
only showing top 5 rows
+-----------+------------------+
|device_type|   avg(cart_value)|
+-----------+------------------+
|    desktop|187.58463829787237|
|     mobile|203.57793991416307|
|     tablet|195.98369294605817|
+-----------+------------------+



## Exercise B — Free-text logs with regex (`app.log`)

These aren't CSV or JSON — just lines of text in two formats:

```
2024-07-03 14:22:11 [INFO] service=api request_id=req-123456 handled request status=200 latency_ms=42
2024-07-03T14:22:11Z level=ERROR svc=payments rid=req-654321 request failed status=500 latency_ms=1201 err=timeout
```

### Goals
1. Read as text: `spark.read.text(...)` → one column named `value`.
2. Use `F.regexp_extract` (and maybe a couple of patterns) to pull out:
   `ts`, `level`, `service`, `request_id`, `status`, `latency_ms`.
3. Cast `status` / `latency_ms` to integers; parse `ts` to timestamp (both formats).
4. Answer:
   - Error rate: `% of lines where level = ERROR`
   - p95-ish latency: what's the 95th percentile of `latency_ms`? (`F.percentile_approx`)
   - Which `service` has the most ERROR lines?

In [4]:
# B0 — peek at raw lines
raw_logs = spark.read.text(f"{DATA_DIR}/app.log")
raw_logs.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------+
|value                                                                                                              |
+-------------------------------------------------------------------------------------------------------------------+
|2024-07-14 15:43:45 [INFO] service=payments request_id=req-410012 handled request status=200 latency_ms=334        |
|2024-07-10T10:04:03Z level=WARN svc=payments rid=req-614764 slow request status=400 latency_ms=712 threshold_ms=800|
|2024-07-04T09:08:25Z level=INFO svc=payments rid=req-552405 handled request status=200 latency_ms=796              |
|2024-07-14 21:23:44 [INFO] service=auth request_id=req-911780 handled request status=401 latency_ms=1255           |
|2024-07-03 17:02:05 [INFO] service=catalog request_id=req-976429 handled request status=500 latency_ms=1206        |
+-------------------------------------------------------

In [5]:
# B1 — YOUR TURN: parse into a structured `logs` DataFrame
# Hints:
#   F.regexp_extract(F.col("value"), r"pattern", 1)
#   Level appears either as [INFO] or level=INFO
#   Service appears as service=... or svc=...
#   request_id as request_id=... or rid=...
#   For timestamps, try coalescing two to_timestamp formats:
#     "yyyy-MM-dd HH:mm:ss" and "yyyy-MM-dd'T'HH:mm:ss'Z'"

#F.regexp_extract('value', r'\d{4}-\d{2}-\d{2}((T)|( ))\d{2}:\d{2}:\d{2}', 0).alias("ts"),
logs = (
    raw_logs.select(F.regexp_extract('value', r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}", 0).alias("ts_raw"),
                    F.regexp_extract('value', r'(?<=\[)[^\]]+(?=\])|(?<=level=)[^&\s]+', 0).alias("level"),
                    F.regexp_extract('value', r'((?<=service=)|(?<=svc=))[^&\s]+', 0).alias("service"),
                    F.regexp_extract('value', r'((?<=request_id=)|(?<=rid=))[^&\s]+', 0).alias("request_id"),
                    F.regexp_extract('value', r'(?<=status=)[^&\s]+', 0).alias("status_code"),
                    F.regexp_extract('value', r'(?<=latency_ms=)[^&\s]+', 0).alias("latency_ms"),
                    F.regexp_extract('value', r'(?<=threshold_ms=)[^&\s]+', 0).alias("threshold_ms"),
                   ).withColumn(
                        "ts",
                        F.coalesce(
                            F.try_to_timestamp(F.col("ts_raw"), F.lit("yyyy-MM-dd HH:mm:ss")),
                            F.try_to_timestamp(F.col("ts_raw"), F.lit("yyyy-MM-dd'T'HH:mm:ss")),
                        ),
                    )
                    .drop("ts_raw")

)
logs.show(10, truncate=False)

+-----+--------+----------+-----------+----------+------------+-------------------+
|level|service |request_id|status_code|latency_ms|threshold_ms|ts                 |
+-----+--------+----------+-----------+----------+------------+-------------------+
|INFO |payments|req-410012|200        |334       |            |2024-07-14 15:43:45|
|WARN |payments|req-614764|400        |712       |800         |2024-07-10 10:04:03|
|INFO |payments|req-552405|200        |796       |            |2024-07-04 09:08:25|
|INFO |auth    |req-911780|401        |1255      |            |2024-07-14 21:23:44|
|INFO |catalog |req-976429|500        |1206      |            |2024-07-03 17:02:05|
|INFO |api     |req-445034|200        |2328      |            |2024-07-12 13:38:27|
|WARN |catalog |req-245305|200        |684       |800         |2024-07-08 01:16:32|
|INFO |catalog |req-169159|200        |808       |            |2024-07-09 04:39:32|
|INFO |catalog |req-597634|200        |1451      |            |2024-07-11 21

In [6]:
# B2 — YOUR TURN: error rate, p95 latency, noisiest service

# Error rate: % of lines where level = ERROR
error_rate = (
    logs.filter(F.col("level") == "ERROR").count()/logs.count()
)
print("ERROR count:", logs.filter(F.col("level") == "ERROR").count())
print("logs count:", logs.count())
print(f"Error Rate: {error_rate * 100} % ",)

ERROR count: 149
logs count: 1500
Error Rate: 9.933333333333334 % 


In [140]:
# p95-ish latency: what's the 95th percentile of latency_ms? (F.percentile_approx)
# df.select(F.percentile_approx("your_column", 0.95).alias("p95")).show()

logs.select(F.percentile_approx("latency_ms", 0.95).alias("p95")).show()

+------+
|   p95|
+------+
|2369.0|
+------+



In [7]:
# Which service has the most ERROR lines?

count_errors = (
    logs.filter(F.col("level") == "ERROR")
        .groupBy(F.col("service"))
        .count().sort(F.desc("count"))
)
count_errors.show()

+--------+-----+
| service|count|
+--------+-----+
|     api|   41|
|payments|   41|
|    auth|   38|
| catalog|   29|
+--------+-----+



## Exercise C — Free-text + multi-value fields (`support_tickets.csv`)

Looks like a normal CSV, but two columns are semi-structured:

- `tags`: pipe-delimited (`billing|urgent|refund`) — sometimes blank
- `body`: prose with buried `customer_id=...` and sometimes `order_id=...`

### Goals
1. Load with `header=True, inferSchema=True`.
2. Explode tags into one row per tag (`F.split` + `F.explode_outer` so blank tags still keep the ticket).
3. Extract `customer_id` and `order_id` from `body` with regex; cast to int (null if missing).
4. Answer:
   - Top 5 tags by ticket count
   - How many tickets mention an `order_id`?
   - Join extracted `customer_id` to `customers.csv` — how many tickets reference a real customer? How many don't?

In [9]:
# C0 — inspect
tickets = spark.read.csv(f"{DATA_DIR}/support_tickets.csv", header=True, inferSchema=True)
customers = spark.read.csv(f"{DATA_DIR}/customers.csv", header=True, inferSchema=True)
tickets.show(10, truncate=80)

+---------+-------------------+-----------------------+--------------------+--------------------------------------------------------------------------------+
|ticket_id|         created_at|                subject|                tags|                                                                            body|
+---------+-------------------+-----------------------+--------------------+--------------------------------------------------------------------------------+
|        1|2024-02-23 22:00:00|App crashes on checkout|                NULL|Customer reported: app crashes on checkout. customer_id=1112 order_id=31910 c...|
|        2|2024-06-03 00:00:00|         Refund status?|              mobile|               Customer reported: refund status?. customer_id=833 channel=email.|
|        3|2024-03-17 19:00:00|     Wrong item shipped|                NULL|Customer reported: wrong item shipped. customer_id=492 order_id=6010 channel=...|
|        4|2024-04-12 04:00:00|App crashes on checko

In [54]:
# C1 — YOUR TURN: structured tickets + exploded tags
# Hints:
#   F.split("tags", r"\|")
#   F.explode_outer(...)
#   F.regexp_extract(F.col("body"), r"customer_id=(\d+)", 1)
#   empty string from regexp_extract → cast carefully ("" won't become null by itself;
#   use F.when(col == "", None) before cast, or nullif)

tickets_clean = tickets.select(
                    F.col("ticket_id"),
                    F.split('tags', r'\|').alias("tags"),
                    F.regexp_extract('body', r'(?<=Customer reported:)\s*(.+?)\.', 0).alias("customer_reported"),
                    F.regexp_extract('body', r'(?<=customer_id=)[^&\s]+', 0).alias("customer_id"),
                    F.regexp_extract('body', r'(?<=order_id=)[^&\s]+', 0).alias("order_id"),
                    F.regexp_extract('body', r'(?<=channel=)[^&\s]+', 0).alias("channel")
)
tickets_clean.show(10)

tickets_by_tag = tickets_clean.withColumn("tag", F.explode_outer("tags")).select(
    F.col("ticket_id"),
    F.col("tag"), 
    F.col("customer_reported"),
    F.col("customer_id"),
    F.when(F.col("order_id") == "", None).otherwise(F.col("order_id")).alias("order_id"),
    F.col("channel")
)
tickets_by_tag.show(15)

+---------+--------------------+--------------------+-----------+--------+-------+
|ticket_id|                tags|   customer_reported|customer_id|order_id|channel|
+---------+--------------------+--------------------+-----------+--------+-------+
|        1|                NULL| app crashes on c...|       1112|   31910|  chat.|
|        2|            [mobile]|     refund status?.|        833|        | email.|
|        3|                NULL| wrong item shipped.|        492|    6010| phone.|
|        4|               [bug]| app crashes on c...|       1144|    6138|  chat.|
|        5|   [billing, urgent]| wrong item shipped.|        397|   19219| phone.|
|        6|       [urgent, vip]|      charged twice.|        412|   15197| phone.|
|        7|[billing, shippin...|      charged twice.|        653|   33745| email.|
|        8|               [vip]|     refund status?.|        602|        |  chat.|
|        9|               [vip]|     refund status?.|       1679|    5349| email.|
|   

In [60]:
# C2 — YOUR TURN: tag counts, order_id coverage, customer match rate
# Top 5 tags by ticket count

top_tags = tickets_by_tag.groupBy(F.col("tag")).agg(F.count_distinct("ticket_id").alias("distinct_tickets")).orderBy(F.desc("distinct_tickets"))
top_tags.show(6) # showed 6 in case of tie

+--------+----------------+
|     tag|distinct_tickets|
+--------+----------------+
|     vip|             192|
|  mobile|             186|
|shipping|             184|
|  refund|             180|
|  urgent|             172|
| billing|             171|
+--------+----------------+
only showing top 6 rows


In [55]:
#How many tickets mention an order_id?

tickets_with_order_id = tickets_by_tag.filter(F.col("order_id").isNotNull())
tickets_with_order_id.select(F.count_distinct("ticket_id").alias("unique_count")).show()


+------------+
|unique_count|
+------------+
|         581|
+------------+



In [ ]:
#Join extracted customer_id to customers.csv — how many tickets reference a real customer? How many don't?

tickets_unique = tickets_by_tag.select(
    "ticket_id", "customer_id"
).dropDuplicates(["ticket_id"])

real = tickets_unique.join(customers, "customer_id", "inner")
fake = tickets_unique.join(customers, "customer_id", "left_anti")
print("real:", real.count())   # 800
print("fake:", fake.count())   # 0


## Stretch goals (optional)

Pick one if you want a harder pass:

1. **Unify formats with a schema.** Define an explicit schema for events (`StructType`) and re-read with `spark.read.schema(...).json(...)`. Compare inferred vs explicit — what broke?
2. **Write curated Parquet.** Save your cleaned `events`, `logs`, and `tickets_clean` under `../output/curated/` as Parquet. Re-read them and confirm queries no longer need regex.
3. **Build a mini incident report.** From `logs`, find ERROR bursts: for each service, count ERRORs per hour (`F.date_trunc("hour", ts)`). Which hour was worst?
4. **End-to-end join.** Tickets → extracted `order_id` → `orders.csv` → `products.csv`. What's the most common product category among tickets that mention an order?

When you're done:

```python
spark.stop()
```

In [ ]:
# Unify formats with a schema. Define an explicit schema for events (StructType) and re-read with spark.read.schema(...).json(...). Compare inferred vs explicit — what broke?

events_schema = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("ts", T.StringType()),
    T.StructField("page", T.StringType()),
    T.StructField("user", T.StructType([
        T.StructField("id", T.LongType()),
        T.StructField("session", T.StringType()),
    ])),
    T.StructField("device", T.StructType([
        T.StructField("type", T.StringType()),
        T.StructField("browser", T.StringType()),
    ])),
    # Use StructType([...]) for props
    T.StructField("props", T.StructType([
        T.StructField("referrer", T.StringType()),
        T.StructField("cart_value", T.StringType()),
    ])),
])

events_schema_string_props = T.StructType([
    T.StructField("event_id", T.StringType()),
    T.StructField("ts", T.StringType()),
    T.StructField("page", T.StringType()),
    T.StructField("user", T.StructType([
        T.StructField("id", T.LongType()),
        T.StructField("session", T.StringType()),
    ])),
    T.StructField("device", T.StructType([
        T.StructField("type", T.StringType()),
        T.StructField("browser", T.StringType()),
    ])),
    # Use StringType() for props
    T.StructField("props", T.StringType()),
])

#Implicit schema
raw_events = spark.read.json(f"{DATA_DIR}/events.jsonl")
raw_events.printSchema()
#Explicit schema using StructType
events_explicit = spark.read.schema(events_schema).json(f"{DATA_DIR}/events.jsonl")
events_explicit.printSchema()

#Explicit schema using StringType
events_explicit2 = spark.read.schema(events_schema_string_props).json(f"{DATA_DIR}/events.jsonl")
events_explicit2.printSchema()

print("implicit null props: ", raw_events.filter(F.col("props").isNull()).count())
print("explcit with struct type: ", events_explicit.filter(F.col("props").isNull()).count())
print("explicit with string type: ", events_explicit2.filter(F.col("props").isNull()).count())

#What broke? 
# Inferred and explicit-string keep all rows (props as string). 
# Explicit struct nulls 103 rows — the ones where props was a JSON string instead of an object. 
# Spark can’t coerce those into a struct, so it drops the field to null.

root
 |-- device: struct (nullable = true)
 |    |-- browser: string (nullable = true)
 |    |-- type: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- props: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- session: string (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- page: string (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- session: string (nullable = true)
 |-- device: struct (nullable = true)
 |    |-- type: string (nullable = true)
 |    |-- browser: string (nullable = true)
 |-- props: struct (nullable = true)
 |    |-- referrer: string (nullable = true)
 |    |-- cart_value: string (nullable = true)

root
 |-- event_id: string (nullable = true)
 |-- ts: string (nullable = true)
 |-- page: string (nullable = true)
 |-- us

In [ ]:
# spark.stop()